In [2]:
!pip install pymupdf chromadb langgraph requests numpy

In [3]:
# ==========================================
# 1. ИМПОРТЫ
# ==========================================
import os
import re
import time
import uuid
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional

import fitz  # PyMuPDF
import requests
import chromadb

from langgraph.graph import StateGraph, START, END

In [4]:
# ==========================================
# 2. НАСТРОЙКИ
# ==========================================
# Если у тебя Ollama или совместимый сервер работает не тут —
# просто поменяй BASE_URL.
BASE_URL = "http://localhost:11434"

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "gpt-oss:120b-cloud"

CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "rag_knowledge"

MAX_CHARS = 1200
OVERLAP = 200
MAX_PAGES_TO_SCAN = 120

TOP_K = 5
REQUEST_TIMEOUT = 300

In [5]:
# ==========================================
# 3. ЛОГ
# ==========================================
def log(*args):
    print("[LOG]", *args)

In [6]:
# ==========================================
# 4. ГЛОБАЛЬНЫЕ ПЕРЕМЕННЫЕ
# ==========================================
GLOBAL_CHROMA_CLIENT = None
GLOBAL_COLLECTION = None
debate_app = None

In [7]:
# ==========================================
# 5. СОСТОЯНИЕ ГРАФА
# ==========================================
class DebateState(TypedDict):
    question: str
    retrieved_context: str
    retrieved_items: List[Dict[str, Any]]
    hypothesis: str
    criticism: str
    evidence: str
    final_answer: str

In [8]:
# ==========================================
# 6. HTTP ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ
# ==========================================
def post_json(url: str, payload: dict, timeout: int = REQUEST_TIMEOUT) -> dict:
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

In [9]:
# ==========================================
# 7. LLM: ЧАТ ЧЕРЕЗ OLLAMA / СОВМЕСТИМЫЙ API
# ==========================================
def ollama_chat(
    system: str,
    user: str,
    model: str = LLM_MODEL,
    options: Optional[dict] = None
) -> str:
    """
    Вызов chat-модели.
    """

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system.strip()},
            {"role": "user", "content": user.strip()}
        ],
        "stream": False
    }

    if options:
        payload["options"] = options

    data = post_json(f"{BASE_URL}/api/chat", payload)

    # Стандартный ответ Ollama
    if "message" in data and isinstance(data["message"], dict):
        return (data["message"].get("content") or "").strip()

    # Запасной вариант
    if "response" in data:
        return (data.get("response") or "").strip()

    return ""

In [10]:
# ==========================================
# 8. EMBEDDINGS: nomic-embed-text
# ==========================================
def get_embedding(text: str, model: str = EMBED_MODEL) -> List[float]:
    """
    Получить embedding для одного текста.
    Сначала пробуем новый endpoint /api/embed,
    если не получилось — fallback на /api/embeddings.
    """
    text = (text or "").strip()
    if not text:
        text = " "

    # Новый endpoint
    try:
        payload = {
            "model": model,
            "input": text
        }
        data = post_json(f"{BASE_URL}/api/embed", payload)

        if "embeddings" in data and data["embeddings"]:
            return data["embeddings"][0]

        if "embedding" in data and data["embedding"]:
            return data["embedding"]
    except Exception:
        pass

    # Старый endpoint
    payload = {
        "model": model,
        "prompt": text
    }
    data = post_json(f"{BASE_URL}/api/embeddings", payload)

    if "embedding" not in data:
        raise ValueError("Embedding не найден в ответе модели")

    return data["embedding"]


def get_embeddings(texts: List[str], model: str = EMBED_MODEL) -> List[List[float]]:
    """
    Получить embeddings для списка текстов.
    Если /api/embed умеет батч — используем батч.
    Иначе fallback: по одному.
    """
    clean_texts = []
    for t in texts:
        t = (t or "").strip()
        if not t:
            t = " "
        clean_texts.append(t)

    # Попытка батча через /api/embed
    try:
        payload = {
            "model": model,
            "input": clean_texts
        }
        data = post_json(f"{BASE_URL}/api/embed", payload)

        if "embeddings" in data and data["embeddings"]:
            return data["embeddings"]
    except Exception:
        pass

    # Fallback по одному
    vectors = []
    for t in clean_texts:
        vectors.append(get_embedding(t, model=model))
    return vectors

In [11]:
# ==========================================
# 9. CHROMA: ИНИЦИАЛИЗАЦИЯ
# ==========================================
def get_chroma_client(path: str = CHROMA_DIR):
    return chromadb.PersistentClient(path=path)


def get_or_create_collection(client=None, name: str = COLLECTION_NAME):
    global GLOBAL_CHROMA_CLIENT

    if client is None:
        client = GLOBAL_CHROMA_CLIENT

    if client is None:
        raise ValueError("Chroma client не инициализирован")

    return client.get_or_create_collection(name=name)

In [12]:
# ==========================================
# 10. CHROMA: ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==========================================
def safe_join(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return "; ".join(str(x) for x in value if x is not None)
    return str(value)


def normalize_metadata_for_chroma(metadata: dict) -> dict:
    metadata = metadata or {}

    return {
        "source_type": str(metadata.get("source_type", "document")),
        "source": safe_join(metadata.get("source")),
        "title": safe_join(metadata.get("title")),
        "chapter": safe_join(metadata.get("chapter")),
        "section": safe_join(metadata.get("section")),
        "subsection": safe_join(metadata.get("subsection")),
        "pages": safe_join(metadata.get("pages")),
        "block_types": safe_join(metadata.get("block_types")),
        "formulas": safe_join(metadata.get("formulas")),
    }


def parse_pages_field(value) -> List[int]:
    if value is None:
        return []

    if isinstance(value, list):
        out = []
        for x in value:
            try:
                out.append(int(x))
            except Exception:
                pass
        return sorted(set(out))

    text = str(value).strip()
    if not text:
        return []

    parts = re.split(r"[;,]\s*", text)
    out = []

    for p in parts:
        p = p.strip()
        if not p:
            continue
        try:
            out.append(int(p))
        except Exception:
            pass

    return sorted(set(out))


def parse_semicolon_field(value) -> List[str]:
    if value is None:
        return []

    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]

    text = str(value).strip()
    if not text:
        return []

    return [x.strip() for x in text.split(";") if x.strip()]

In [13]:
# ==========================================
# 11. СОЗДАТЬ НОВУЮ ПУСТУЮ БАЗУ С НУЛЯ
# ==========================================
def reset_knowledge_base(
    chroma_dir: str = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME
):
    global GLOBAL_CHROMA_CLIENT, GLOBAL_COLLECTION, debate_app

    client = get_chroma_client(chroma_dir)

    try:
        client.delete_collection(collection_name)
        log(f"Старая коллекция '{collection_name}' удалена.")
    except Exception:
        log(f"Коллекция '{collection_name}' ещё не существовала.")

    collection = client.get_or_create_collection(name=collection_name)

    GLOBAL_CHROMA_CLIENT = client
    GLOBAL_COLLECTION = collection
    debate_app = build_debate_graph()

    log("Создана новая пустая база.")
    return collection


def load_ready_knowledge_base(
    chroma_dir: str = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME
):
    global GLOBAL_CHROMA_CLIENT, GLOBAL_COLLECTION, debate_app

    client = get_chroma_client(chroma_dir)
    collection = client.get_or_create_collection(name=collection_name)

    GLOBAL_CHROMA_CLIENT = client
    GLOBAL_COLLECTION = collection
    debate_app = build_debate_graph()

    log(f"База загружена. Записей: {collection.count()}")
    return collection

In [14]:
# ==========================================
# 12. ЭВРИСТИКИ ДЛЯ БЛОКОВ
# ==========================================
def looks_like_math(text: str) -> bool:
    if not text:
        return False

    text = text.strip()
    if not text:
        return False

    math_tokens = [
        "=", "∫", "π", "δ", "√", "∞", "exp", "sin", "cos", "tan",
        "dx", "dy", "dz", "sinc", "rect", "fft", "∑", "−", "±",
        "\\", "^", "_", "{", "}", "[", "]"
    ]

    if any(tok in text for tok in math_tokens):
        return True

    if re.search(r"\(\d+(\.\d+)*\)\s*$", text):
        return True

    special_count = sum(ch in "=+-*/^_[]{}()<>|\\∫πδ√∞" for ch in text)
    if special_count >= 4:
        return True

    if re.search(r"[A-Za-zА-Яа-я]\s*=\s*[A-Za-zА-Яа-я0-9]", text):
        return True

    return False


def is_bold_font(font_name: str, flags: int = 0) -> bool:
    font_name = (font_name or "").lower()

    if "bold" in font_name:
        return True

    if isinstance(flags, int) and (flags & 16):
        return True

    return False


def classify_block(text: str, max_size: float, avg_size: float, bold_ratio: float) -> str:
    text = (text or "").strip()

    if not text:
        return "empty"

    if looks_like_math(text):
        return "formula"

    if max_size >= 15 and bold_ratio >= 0.5 and len(text) <= 180:
        return "heading_1"

    if max_size >= 13 and bold_ratio >= 0.4 and len(text) <= 180:
        return "heading_2"

    if max_size >= 11.5 and bold_ratio >= 0.35 and len(text) <= 200:
        return "heading_3"

    return "text"

In [15]:
# ==========================================
# 13. ИЗВЛЕЧЕНИЕ БЛОКОВ СО СТРАНИЦЫ
# ==========================================
def extract_text_blocks(page, drop_headers=True, drop_footers=True):
    blocks = []

    page_height = page.rect.height
    raw_blocks = page.get_text("dict")["blocks"]

    for b in raw_blocks:
        if "lines" not in b:
            continue

        x0, y0, x1, y1 = b["bbox"]

        # Верхний колонтитул
        if drop_headers and y0 < page_height * 0.06:
            continue

        # Нижний колонтитул
        if drop_footers and y1 > page_height * 0.92:
            continue

        spans_info = []
        texts = []

        for line in b["lines"]:
            for span in line["spans"]:
                span_text = (span.get("text") or "").strip()
                if not span_text:
                    continue

                texts.append(span_text)
                spans_info.append({
                    "text": span_text,
                    "size": float(span.get("size", 0)),
                    "font": span.get("font", ""),
                    "flags": int(span.get("flags", 0)),
                    "is_bold": is_bold_font(span.get("font", ""), int(span.get("flags", 0)))
                })

        text = " ".join(texts).strip()
        if not text:
            continue

        sizes = [s["size"] for s in spans_info] if spans_info else [0.0]
        max_size = max(sizes) if sizes else 0.0
        avg_size = sum(sizes) / len(sizes) if sizes else 0.0
        bold_ratio = (
            sum(1 for s in spans_info if s["is_bold"]) / len(spans_info)
            if spans_info else 0.0
        )

        block_type = classify_block(
            text=text,
            max_size=max_size,
            avg_size=avg_size,
            bold_ratio=bold_ratio
        )

        blocks.append({
            "text": text,
            "bbox": b["bbox"],
            "page_num": page.number,
            "type": block_type,
            "max_size": max_size,
            "avg_size": avg_size,
            "bold_ratio": bold_ratio,
            "spans": spans_info,
        })

    return blocks

In [16]:
# ==========================================
# 14. СПИСОК РАБОЧИХ СТРАНИЦ
# ==========================================
def get_working_page_numbers(doc, max_pages_to_scan: Optional[int] = None):
    total_pages = len(doc)

    if max_pages_to_scan is None:
        max_pages_to_scan = total_pages

    max_pages_to_scan = min(max_pages_to_scan, total_pages)
    return list(range(max_pages_to_scan))

In [17]:
# ==========================================
# 15. ИЗВЛЕЧЕНИЕ БЛОКОВ СО ВСЕХ СТРАНИЦ
# ==========================================
def extract_blocks_from_working_pages(doc, working_pages):
    all_blocks = []

    log("Начинаю извлечение блоков...")

    for n, page_num in enumerate(working_pages, start=1):
        page = doc[page_num]
        page_blocks = extract_text_blocks(page)

        for b in page_blocks:
            b["page_num"] = page_num

        all_blocks.extend(page_blocks)
        log(f"[{n}/{len(working_pages)}] стр. {page_num + 1} -> блоков: {len(page_blocks)}")

    log(f"Всего извлечено блоков: {len(all_blocks)}")
    return all_blocks

In [18]:
# ==========================================
# 16. СБОРКА ЧАНКОВ
# ==========================================
def chunk_blocks_with_metadata(
    blocks,
    max_chars: int = MAX_CHARS,
    overlap: int = OVERLAP,
    source_name: str = "unknown_source"
):
    chunks = []

    current_chapter = None
    current_section = None
    current_subsection = None

    current_text_parts = []
    current_pages = []
    current_formulas = []
    current_block_types = []

    def flush_chunk():
        nonlocal current_text_parts, current_pages, current_formulas, current_block_types

        text = "\n".join(part for part in current_text_parts if part.strip()).strip()
        if not text:
            return

        chunks.append({
            "text": text,
            "pages": sorted(set(current_pages)),
            "metadata": {
                "source_type": "document",
                "source": source_name,
                "title": "",
                "chapter": current_chapter,
                "section": current_section,
                "subsection": current_subsection,
                "block_types": sorted(set(current_block_types)),
                "formulas": current_formulas.copy(),
            }
        })

        # overlap
        if overlap > 0 and len(text) > overlap:
            tail = text[-overlap:].strip()
            current_text_parts = [tail] if tail else []
        else:
            current_text_parts = []

        current_pages = []
        current_formulas = []
        current_block_types = []

    log("Начинаю сборку чанков...")

    for b in blocks:
        text = (b.get("text") or "").strip()
        block_type = b.get("type", "text")
        page_num = b.get("page_num")

        if not text:
            continue

        if block_type == "heading_1":
            flush_chunk()
            current_chapter = text
            current_section = None
            current_subsection = None
            continue

        if block_type == "heading_2":
            flush_chunk()
            current_section = text
            current_subsection = None
            continue

        if block_type == "heading_3":
            flush_chunk()
            current_subsection = text
            continue

        if block_type == "formula":
            formula_text = f"[FORMULA]\n{text}\n[/FORMULA]"
            candidate_text = "\n".join(current_text_parts + [formula_text]).strip()

            if len(candidate_text) > max_chars and current_text_parts:
                flush_chunk()

            current_text_parts.append(formula_text)
            current_formulas.append(text)
            current_block_types.append("formula")

            if page_num is not None:
                current_pages.append(page_num + 1)

            continue

        candidate_text = "\n".join(current_text_parts + [text]).strip()

        if len(candidate_text) > max_chars and current_text_parts:
            flush_chunk()

        current_text_parts.append(text)
        current_block_types.append("text")

        if page_num is not None:
            current_pages.append(page_num + 1)

    flush_chunk()

    log(f"Чанков собрано: {len(chunks)}")
    return chunks

In [19]:
# ==========================================
# 17. PDF -> CHUNKS
# ==========================================
def build_chunks_from_pdf(
    doc,
    source_name: str = "book.pdf",
    max_pages_to_scan: Optional[int] = MAX_PAGES_TO_SCAN,
    max_chars: int = MAX_CHARS,
    overlap: int = OVERLAP
):
    working_pages = get_working_page_numbers(doc, max_pages_to_scan=max_pages_to_scan)
    log("Рабочих страниц:", len(working_pages))

    all_blocks = extract_blocks_from_working_pages(doc, working_pages)
    log("Всего блоков:", len(all_blocks))

    chunks = chunk_blocks_with_metadata(
        blocks=all_blocks,
        max_chars=max_chars,
        overlap=overlap,
        source_name=source_name
    )

    log("Всего чанков:", len(chunks))
    return chunks

In [20]:
# ==========================================
# 18. ДОБАВЛЕНИЕ ЧАНКОВ В CHROMA
# ==========================================
def add_chunks_to_chroma(chunks: List[dict], collection=None):
    global GLOBAL_COLLECTION

    if collection is None:
        collection = GLOBAL_COLLECTION

    if collection is None:
        raise ValueError("Коллекция Chroma не загружена")

    if not chunks:
        log("Пустой список chunks. Нечего добавлять.")
        return 0

    ids = []
    documents = []
    metadatas = []
    embeddings = []

    valid_count = 0

    for chunk in chunks:
        text = (chunk.get("text") or "").strip()
        if not text:
            continue

        metadata = chunk.get("metadata", {}) or {}
        if "pages" not in metadata:
            metadata["pages"] = chunk.get("pages", [])

        ids.append(f"chunk_{uuid.uuid4().hex}")
        documents.append(text)
        metadatas.append(normalize_metadata_for_chroma(metadata))
        embeddings.append(get_embedding(text))

        valid_count += 1

        if valid_count % 10 == 0:
            log(f"Подготовлено чанков: {valid_count}")

    if not ids:
        log("Нет валидных чанков для записи.")
        return 0

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )

    log(f"В Chroma добавлено чанков: {len(ids)}")
    return len(ids)

In [21]:
# ==========================================
# 19. СОЗДАНИЕ НОВОЙ БАЗЫ ИЗ PDF С НУЛЯ
# ==========================================
def build_knowledge_base_from_pdf(
    pdf_path: str,
    source_name: Optional[str] = None,
    chroma_dir: str = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
    max_pages_to_scan: int = MAX_PAGES_TO_SCAN,
    max_chars: int = MAX_CHARS,
    overlap: int = OVERLAP,
    reset_db: bool = True
):
    global GLOBAL_CHROMA_CLIENT, GLOBAL_COLLECTION, debate_app

    pdf_path = str(pdf_path)

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF не найден: {pdf_path}")

    if source_name is None:
        source_name = Path(pdf_path).stem

    if reset_db:
        reset_knowledge_base(chroma_dir=chroma_dir, collection_name=collection_name)
    else:
        load_ready_knowledge_base(chroma_dir=chroma_dir, collection_name=collection_name)

    log("=" * 80)
    log("СТАРТ ПОСТРОЕНИЯ БАЗЫ ИЗ PDF")
    log("PDF:", pdf_path)
    log("SOURCE:", source_name)
    log("=" * 80)

    start_time = time.time()

    doc = fitz.open(pdf_path)
    log("Всего страниц в PDF:", len(doc))

    chunks = build_chunks_from_pdf(
        doc=doc,
        source_name=source_name,
        max_pages_to_scan=max_pages_to_scan,
        max_chars=max_chars,
        overlap=overlap
    )

    if not chunks:
        log("Чанки не созданы.")
        return GLOBAL_COLLECTION

    log("Пример первого чанка:")
    log("pages:", chunks[0].get("pages"))
    log("metadata:", chunks[0].get("metadata"))
    log("text preview:", chunks[0].get("text", "")[:300])

    added = add_chunks_to_chroma(chunks, collection=GLOBAL_COLLECTION)

    debate_app = build_debate_graph()

    elapsed = time.time() - start_time

    log("=" * 80)
    log("БАЗА ГОТОВА")
    log("Добавлено чанков:", added)
    log("Всего записей в коллекции:", GLOBAL_COLLECTION.count())
    log("Время:", round(elapsed / 60, 2), "мин")
    log("=" * 80)

    return GLOBAL_COLLECTION

In [22]:
# ==========================================
# 20. ДОБАВИТЬ ЕЩЁ ОДИН PDF В УЖЕ ГОТОВУЮ БАЗУ
# ==========================================
def add_pdf_to_existing_knowledge_base(
    pdf_path: str,
    source_name: Optional[str] = None,
    max_pages_to_scan: int = MAX_PAGES_TO_SCAN,
    max_chars: int = MAX_CHARS,
    overlap: int = OVERLAP
):
    global GLOBAL_COLLECTION, debate_app

    if GLOBAL_COLLECTION is None:
        load_ready_knowledge_base()

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF не найден: {pdf_path}")

    if source_name is None:
        source_name = Path(pdf_path).stem

    log("=" * 80)
    log("ДОБАВЛЕНИЕ PDF В БАЗУ")
    log("PDF:", pdf_path)
    log("SOURCE:", source_name)
    log("=" * 80)

    start_time = time.time()

    doc = fitz.open(pdf_path)

    chunks = build_chunks_from_pdf(
        doc=doc,
        source_name=source_name,
        max_pages_to_scan=max_pages_to_scan,
        max_chars=max_chars,
        overlap=overlap
    )

    added = add_chunks_to_chroma(chunks, collection=GLOBAL_COLLECTION)

    debate_app = build_debate_graph()

    elapsed = time.time() - start_time

    log("=" * 80)
    log("PDF ДОБАВЛЕН")
    log("Добавлено чанков:", added)
    log("Всего записей в коллекции:", GLOBAL_COLLECTION.count())
    log("Время:", round(elapsed / 60, 2), "мин")
    log("=" * 80)

    return GLOBAL_COLLECTION

In [23]:
# ==========================================
# 21. ПОИСК ПО БАЗЕ
# ==========================================
def search_in_db(
    query: str,
    collection=None,
    k: int = TOP_K,
    where: Optional[dict] = None
):
    global GLOBAL_COLLECTION

    if collection is None:
        collection = GLOBAL_COLLECTION

    if collection is None:
        print("База не загружена.")
        return []

    log(f"Поиск по базе. Запрос: {query}")

    q_emb = get_embedding(query)

    try:
        result = collection.query(
            query_embeddings=[q_emb],
            n_results=k,
            where=where,
            include=["documents", "metadatas", "distances"]
        )
    except Exception as e:
        print(f"Ошибка поиска: {type(e).__name__}: {e}")
        return []

    documents = result.get("documents", [[]])
    metadatas = result.get("metadatas", [[]])
    distances = result.get("distances", [[]])
    ids = result.get("ids", [[]])

    if not documents or not documents[0]:
        log("Ничего не найдено.")
        return []

    out = []

    for i, (doc_text, meta, dist, item_id) in enumerate(zip(
        documents[0],
        metadatas[0],
        distances[0] if distances else [None] * len(documents[0]),
        ids[0]
    )):
        meta = meta or {}

        out.append({
            "idx": i,
            "doc_id": item_id,
            "distance": float(dist) if dist is not None else None,
            "text": doc_text or "",
            "pages": parse_pages_field(meta.get("pages", "")),
            "source_type": meta.get("source_type", "document"),
            "source": meta.get("source", ""),
            "title": meta.get("title", ""),
            "chapter": meta.get("chapter", ""),
            "section": meta.get("section", ""),
            "subsection": meta.get("subsection", ""),
            "formulas": parse_semicolon_field(meta.get("formulas", "")),
            "block_types": parse_semicolon_field(meta.get("block_types", "")),
        })

    log(f"Найдено результатов: {len(out)}")
    return out

In [24]:
# ==========================================
# 22. СБОРКА КОНТЕКСТА ИЗ РЕЗУЛЬТАТОВ ПОИСКА
# ==========================================
def build_context_from_results(results: List[Dict[str, Any]]) -> str:
    if not results:
        return ""

    parts = []

    for i, item in enumerate(results, start=1):
        source = item.get("source") or "unknown_source"
        chapter = item.get("chapter") or "—"
        section = item.get("section") or "—"
        subsection = item.get("subsection") or "—"
        pages = item.get("pages", [])
        distance = item.get("distance", 0.0)
        text = item.get("text", "").strip()
        formulas = item.get("formulas", [])

        page_str = ", ".join(map(str, pages)) if pages else "—"

        formulas_block = "нет"
        if formulas:
            formulas_block = "\n".join(f"- {f}" for f in formulas[:5])

        part = (
            f"[ФРАГМЕНТ {i}]\n"
            f"Источник: {source}\n"
            f"Глава: {chapter}\n"
            f"Раздел: {section}\n"
            f"Подраздел: {subsection}\n"
            f"Страницы: {page_str}\n"
            f"Distance: {distance:.4f}\n"
            f"Формулы:\n{formulas_block}\n"
            f"Текст:\n{text}\n"
        )
        parts.append(part)

    return "\n\n".join(parts)

In [25]:
# ==========================================
# 23. ФОРМАТИРОВАНИЕ БЛОКА ИСТОЧНИКОВ
# ==========================================
def format_sources_block(items: List[Dict[str, Any]]) -> str:
    if not items:
        return "ИСТОЧНИКИ\n\nИсточники не указаны."

    lines = ["ИСТОЧНИКИ", ""]

    for i, item in enumerate(items, start=1):
        source = item.get("source") or "unknown_source"
        title = item.get("title") or "—"
        chapter = item.get("chapter") or "—"
        section = item.get("section") or "—"
        subsection = item.get("subsection") or "—"
        pages = item.get("pages", [])
        page_str = ", ".join(map(str, pages)) if pages else "—"

        lines.append(
            f"[Фрагмент {i}] Источник: {source} | Заголовок: {title} | "
            f"Глава: {chapter} | Раздел: {section} | Подраздел: {subsection} | Страницы: {page_str}"
        )

    return "\n".join(lines)

In [26]:
# ==========================================
# 24. УЗЕЛ: ИЗВЛЕЧЕНИЕ КОНТЕКСТА
# ==========================================
def retrieve_context_node(state: DebateState) -> dict:
    question = state["question"]
    log("Debate node: retrieve_context")

    results = search_in_db(question, k=TOP_K)

    if not results:
        return {
            "retrieved_items": [],
            "retrieved_context": ""
        }

    context = build_context_from_results(results)

    return {
        "retrieved_items": results,
        "retrieved_context": context
    }

In [27]:
# ==========================================
# 25. УЗЕЛ: ГИПОТЕЗА
# ==========================================
def build_hypothesis_node(state: DebateState) -> dict:
    log("Debate node: build_hypothesis")

    question = state["question"]
    context = state.get("retrieved_context", "")

    if not context.strip():
        return {
            "hypothesis": "Гипотеза не может быть сформулирована, потому что в базе не найден релевантный контекст."
        }

    system = """
Ты аналитик RAG-системы.

Твоя задача:
- прочитать вопрос;
- прочитать контекст из базы;
- сформулировать предварительную гипотезу;
- опираться только на контекст;
- не выдумывать факты;
- если данных мало, прямо скажи об этом.

Пиши на русском языке.
Выводи только гипотезу.
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Сформулируй предварительную гипотезу строго на основе этого контекста.
"""

    hypothesis = ollama_chat(system=system, user=user)
    return {"hypothesis": hypothesis}

In [28]:
# ==========================================
# 26. УЗЕЛ: КРИТИКА
# ==========================================
def critic_review_node(state: DebateState) -> dict:
    log("Debate node: critic_review")

    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")

    if not context.strip():
        return {
            "criticism": "Критика невозможна, потому что в базе не найден релевантный контекст."
        }

    system = """
Ты критик-аналитик.

Твоя задача:
- проверить гипотезу на слабые места;
- указать, что в ней подтверждается контекстом;
- указать, что подтверждается слабо;
- указать противоречия;
- указать, каких данных не хватает;
- использовать только контекст из базы.

Пиши на русском языке.
Выводи только критический разбор.
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Проведи критический анализ гипотезы.
"""

    criticism = ollama_chat(system=system, user=user)
    return {"criticism": criticism}

In [29]:
# ==========================================
# 27. УЗЕЛ: ПРОВЕРКА АРГУМЕНТОВ
# ==========================================
def evidence_check_node(state: DebateState) -> dict:
    log("Debate node: evidence_check")

    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")
    criticism = state.get("criticism", "")

    if not context.strip():
        return {
            "evidence": "Проверка аргументов невозможна, потому что в базе не найден релевантный контекст."
        }

    system = """
Ты аналитик доказательств.

Твоя задача:
- выделить, что в контексте поддерживает гипотезу;
- выделить, что в контексте ослабляет или опровергает гипотезу;
- не придумывать факты;
- использовать только переданный контекст.

Пиши на русском языке.

Формат:
Подтверждает:
- ...

Ослабляет или опровергает:
- ...
"""

    user = f"""
Вопрос:
{question}

Контекст:
{context}

Гипотеза:
{hypothesis}

Критика:
{criticism}

Выдели подтверждающие и ослабляющие аргументы.
"""

    evidence = ollama_chat(system=system, user=user)
    return {"evidence": evidence}

In [30]:
# ==========================================
# 28. УЗЕЛ: ФИНАЛЬНЫЙ ВЫВОД
# ==========================================
def final_conclusion_node(state: DebateState) -> dict:
    log("Debate node: final_conclusion")

    question = state["question"]
    context = state.get("retrieved_context", "")
    hypothesis = state.get("hypothesis", "")
    criticism = state.get("criticism", "")
    evidence = state.get("evidence", "")
    retrieved_items = state.get("retrieved_items", [])

    system = """
Ты финальный аналитик debate-RAG.

Твоя задача:
- ответить на вопрос пользователя;
- учесть найденный контекст;
- учесть гипотезу;
- учесть критику;
- учесть проверку аргументов;
- сделать логичный, аккуратный и понятный итоговый вывод.

Правила:
- не придумывай факты вне контекста;
- если данных не хватает, прямо скажи об этом;
- пиши на русском языке;
- если есть формулы, оставляй их в LaTeX.

Структура ответа:
1. Суть вопроса
2. Основной анализ
3. Ограничения и сомнения
4. Итоговый вывод
"""

    user = f"""
ВОПРОС:
{question}

КОНТЕКСТ:
{context}

ГИПОТЕЗА:
{hypothesis}

КРИТИКА:
{criticism}

ПРОВЕРКА АРГУМЕНТОВ:
{evidence}

Сделай финальный вывод.
"""

    final_answer = ollama_chat(system=system, user=user)
    sources_block = format_sources_block(retrieved_items)

    full_answer = f"""{final_answer}

{sources_block}
"""

    return {"final_answer": full_answer}

In [31]:
# ==========================================
# 29. СБОРКА LANGGRAPH
# ==========================================
def build_debate_graph():
    log("Собираю debate graph...")

    graph = StateGraph(DebateState)

    graph.add_node("retrieve_context", retrieve_context_node)
    graph.add_node("build_hypothesis", build_hypothesis_node)
    graph.add_node("critic_review", critic_review_node)
    graph.add_node("evidence_check", evidence_check_node)
    graph.add_node("final_conclusion", final_conclusion_node)

    graph.add_edge(START, "retrieve_context")
    graph.add_edge("retrieve_context", "build_hypothesis")
    graph.add_edge("build_hypothesis", "critic_review")
    graph.add_edge("critic_review", "evidence_check")
    graph.add_edge("evidence_check", "final_conclusion")
    graph.add_edge("final_conclusion", END)

    app = graph.compile()
    log("Debate graph готов.")
    return app

In [32]:
# ==========================================
# 30. ЗАПУСК ВОПРОСА К RAG
# ==========================================
def ask_debate_rag(question: str) -> str:
    global debate_app

    if debate_app is None:
        raise ValueError("Debate graph не собран. Сначала создай или загрузи базу.")

    log("Запуск debate RAG...")

    state: DebateState = {
        "question": question,
        "retrieved_context": "",
        "retrieved_items": [],
        "hypothesis": "",
        "criticism": "",
        "evidence": "",
        "final_answer": ""
    }

    result = debate_app.invoke(state)
    return result["final_answer"]

In [33]:
# ==========================================
# 31. СПИСОК ИСТОЧНИКОВ В БАЗЕ
# ==========================================
def list_books(limit: int = 5000):
    global GLOBAL_COLLECTION

    if GLOBAL_COLLECTION is None:
        print("База не загружена.")
        return

    try:
        data = GLOBAL_COLLECTION.get(include=["metadatas"], limit=limit)
    except Exception as e:
        print(f"Ошибка чтения коллекции: {e}")
        return

    metadatas = data.get("metadatas", []) or []

    books = sorted({
        (m or {}).get("source", "unknown_source")
        for m in metadatas
    })

    print("\nИсточники в базе:\n")
    for b in books:
        print("-", b)

    print("\nВсего источников:", len(books))

In [34]:
# ==========================================
# 32. ПРОСМОТР РЕЗУЛЬТАТОВ ПОИСКА
# ==========================================
def print_search_results(results: List[Dict[str, Any]], max_chars: int = 800):
    if not results:
        print("Ничего не найдено.")
        return

    for i, r in enumerate(results, start=1):
        print("=" * 100)
        print(f"[{i}] SOURCE:", r.get("source"))
        print("TITLE:", r.get("title"))
        print("CHAPTER:", r.get("chapter"))
        print("SECTION:", r.get("section"))
        print("SUBSECTION:", r.get("subsection"))
        print("PAGES:", r.get("pages"))
        print("DISTANCE:", r.get("distance"))
        print("-" * 100)
        print((r.get("text") or "")[:max_chars])
        print()

In [35]:
# ==========================================
# 33. БЫСТРЫЙ СТАРТ: СОЗДАТЬ НОВУЮ БАЗУ С НУЛЯ
# ==========================================
# collection = build_knowledge_base_from_pdf(
#     pdf_path=r"G:\Мой диск\Книги\Новая папка\0370 Steven G Johnson - Photonic Crystals From Theory to Practice - 2001.pdf",
#     source_name="0370 Steven G Johnson - Photonic Crystals From Theory to Practice - 2001",
#     reset_db=True
# )

In [36]:
# ==========================================
# 34. БЫСТРЫЙ СТАРТ: ДОБАВИТЬ ЕЩЁ PDF
# ==========================================
# Пример:
# add_pdf_to_existing_knowledge_base(
#     pdf_path="second_book.pdf",
#     source_name="SecondBook"
# )

In [ ]:
# ==========================================
# 35. БЫСТРЫЙ СТАРТ: ЗАГРУЗИТЬ ГОТОВУЮ БАЗУ
# ==========================================

load_ready_knowledge_base()

In [ ]:
# ==========================================
# 36. БЫСТРЫЙ СТАРТ: ПРОВЕРИТЬ ПОИСК
# ==========================================
# Пример:
# results = search_in_db("интерференция света", k=5)
# print_search_results(results)

In [ ]:
# ==========================================
# 37. БЫСТРЫЙ СТАРТ: ЗАДАТЬ ВОПРОС
# ==========================================
answer = ask_debate_rag("Предложи технологию, чтобы сделать невидимым человеческому глазу предмет размером несколько метров с помощью метаматериалов")
print(answer)